In [13]:
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from tensorflow import keras
from tensorflow.keras import layers

DATA_PATH = Path('Churn_Modelling.csv')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)


In [14]:
df = pd.read_csv(DATA_PATH).copy()
df['BalanceZero'] = (df['Balance'] == 0).astype(int)
df['ProductsPerTenure'] = df['NumOfProducts'] / (df['Tenure'] + 1)
df['SalaryPerProduct'] = df['EstimatedSalary'] / (df['NumOfProducts'] + 1)

y = df['Exited'].astype(int)
X = df.drop(columns=['Exited', 'RowNumber', 'CustomerId', 'Surname'])
feature_columns = X.columns.tolist()

categorical_features = ['Gender', 'Geography']
numeric_features = [column for column in X.columns if column not in categorical_features]

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=RANDOM_STATE, stratify=y_train_full
)

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]), numeric_features),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]), categorical_features),
])

X_train_prepared = preprocessor.fit_transform(X_train)
X_val_prepared = preprocessor.transform(X_val)
X_test_prepared = preprocessor.transform(X_test)

class_weights = compute_class_weight(class_weight='balanced', classes=np.array([0, 1]), y=y_train)
class_weight = {0: class_weights[0], 1: class_weights[1]}

model = keras.Sequential([
    layers.Input(shape=(X_train_prepared.shape[1],)),
    layers.Dense(96, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(48, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(24, activation='relu'),
    layers.Dense(1, activation='sigmoid'),
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')],
)

history = model.fit(
    X_train_prepared,
    y_train,
    validation_data=(X_val_prepared, y_val),
    epochs=120,
    batch_size=64,
    verbose=0,
    class_weight=class_weight,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(patience=6, factor=0.5, min_lr=1e-5),
    ],
)

val_probabilities = model.predict(X_val_prepared, verbose=0).ravel()
thresholds = np.linspace(0.3, 0.75, 46)
best_threshold, best_val_accuracy = max(
    ((threshold, accuracy_score(y_val, (val_probabilities >= threshold).astype(int))) for threshold in thresholds),
    key=lambda item: item[1],
)

test_probabilities = model.predict(X_test_prepared, verbose=0).ravel()
test_predictions = (test_probabilities >= best_threshold).astype(int)
print({'best_threshold': best_threshold, 'val_accuracy': best_val_accuracy, 'test_accuracy': accuracy_score(y_test, test_predictions), 'test_auc': roc_auc_score(y_test, test_probabilities)})
print(classification_report(y_test, test_predictions))
print(confusion_matrix(y_test, test_predictions))

baseline = RandomForestClassifier(n_estimators=400, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1)
baseline.fit(X_train_prepared, y_train)
feature_names = preprocessor.get_feature_names_out()
importance_rows = []
for feature_name, importance in zip(feature_names, baseline.feature_importances_):
    if feature_name.startswith('num__'):
        importance_rows.append((feature_name.replace('num__', ''), importance))
    else:
        encoded_name = feature_name.replace('cat__', '')
        matched_group = next(feature for feature in categorical_features if encoded_name.startswith(feature + '_'))
        importance_rows.append((matched_group, importance))
importance_df = pd.DataFrame(importance_rows, columns=['feature', 'importance']).groupby('feature', as_index=False)['importance'].sum().sort_values('importance', ascending=False)
top_feature = importance_df.iloc[0]['feature']
print('Top feature:', top_feature)
importance_df.head(10)


{'best_threshold': np.float64(0.73), 'val_accuracy': 0.851875, 'test_accuracy': 0.865, 'test_auc': np.float64(0.8588357232425028)}
              precision    recall  f1-score   support

           0       0.89      0.95      0.92      1593
           1       0.73      0.54      0.62       407

    accuracy                           0.86      2000
   macro avg       0.81      0.74      0.77      2000
weighted avg       0.86      0.86      0.86      2000

[[1510   83]
 [ 187  220]]
Top feature: Age


,feature,importance
0,Age,0.237659
9,NumOfProducts,0.132598
1,Balance,0.100748
11,SalaryPerProduct,0.098929
3,CreditScore,0.094537
4,EstimatedSalary,0.090510
10,ProductsPerTenure,0.057565
8,IsActiveMember,0.048293
12,Tenure,0.044746
6,Geography,0.044679
